# Retrieval-Augmented Generation (RAG)
## Information Retrieval and Search
### Keyword Search

In [ ]:
import os
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("gpreda/bbc-news")
news_data = pd.read_csv(os.path.join(path, "bbc_news.csv"))
news_data.head()

In [ ]:
import bm25s
corpus = list(news_data["title"] + " " + news_data["description"])
retriever = bm25s.BM25(corpus=corpus)

corpus_tokens = bm25s.tokenize(corpus)
print(f"Documents: {len(corpus_tokens.ids)}, Vocab: {len(corpus_tokens.vocab)}")
retriever.index(corpus_tokens)

query = "Retrieval augmented generation"
query_tokens = bm25s.tokenize(query)
docs, scores = retriever.retrieve(query_tokens, k=3)
print(f"Best result (score: {scores[0, 0]:.2f}): {docs[0, 0]}")

In [ ]:
# {24538: 'augmented', 3129: 'generation'}
id_query = {corpus_tokens.vocab[q]: q for q in list(query_tokens.vocab) if q in corpus_tokens.vocab}

# [19281, 17907, 18662]
id_doc = [corpus.index(d) for d in docs[0]]

count = pd.DataFrame({d: {id_query[t]: corpus_tokens.ids[d].count(t) for t in id_query} for d in id_doc}).T
score = pd.Series(scores[0], index=id_doc, name="score")

In [ ]:
count.join(news_data).join(score)

### Semantic Search

In [ ]:
from sentence_transformers import SentenceTransformer
import joblib

'''
all-MiniLM-L6-v2            8 minutes
paraphrase-MiniLM-L3-v2     5 minutes
'''
model_name = "paraphrase-MiniLM-L3-v2" # minutes
model = SentenceTransformer(model_name)
data = list(news_data["title"] + " " + news_data["description"])
embeddings = model.encode(data)

joblib.dump(embeddings, model_name.split("/")[-1] + ".joblib")

In [ ]:
embeddings = None
embeddings = joblib.load(model_name + ".joblib")